# Notebook per Fine-Tuning Classificazione (SpectraMAE / HybridMAE)

Questo notebook implementa il fine-tuning per task di classificazione.
Supporta sia l'architettura **SpectraTransformer** sia **HybridCNNTransformer** e modalità:
- `scratch` — training da zero.
- `linear_probe` — solo la testa di classificazione (encoder congelato).
- `finetune` — warm-up con Linear Probe, poi fine-tuning completo a LR ridotto.

Tutti gli iperparametri vengono dal file `.yaml` (via `EXP_CONFIG_FILE`).

In [ ]:
# --- CELL 1: SETUP AND IMPORTS ---
import sys
import os
import yaml
import json
import numpy as np
import torch

from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import classification_report, accuracy_score, f1_score, roc_auc_score

PROJECT_PATH = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_PATH not in sys.path:
    sys.path.insert(0, PROJECT_PATH)
os.chdir(PROJECT_PATH)
print(f"Working dir: {os.getcwd()}")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

In [ ]:
# --- CELL 2: LOAD CONFIG AND OUTPUT DIR ---
import shutil

# Default: HybridCNNTransformer su IBD
# Per architetture CNN/Transformer, cambia il path o imposta EXP_CONFIG_FILE
config_file = os.environ.get(
    'EXP_CONFIG_FILE',
    'configs/classification/IBD/Hybrid/exp_01_hybrid_finetune.yaml'
)

assert os.path.exists(config_file), f'Config not found: {config_file}'

with open(config_file, 'r') as f:
    config = yaml.safe_load(f)

ARCHITECTURE = config.get('model', {}).get('architecture', 'CNN')
print(f'🏗️  Architettura selezionata: {ARCHITECTURE}')

SEED = 42
split_seed = config.get('split', {}).get('seed', None)
if split_seed is not None:
    SEED = int(split_seed)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(SEED)
        torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f'Seed from YAML active: {SEED}')

output_dir = config_file.replace('configs', 'experiments').replace('.yaml', '')
os.makedirs(output_dir, exist_ok=True)
shutil.copy(config_file, os.path.join(output_dir, 'config_usato.yaml'))

print(f"Experiment: {config.get('experiment_name', 'unknown')}")
print(f"Config: {config_file}")
print(f"Output dir: {output_dir}")

In [ ]:
# --- CELL 3: LOAD DATA AND PREPARE SPLITS ---
import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader
from src.data.mat_loader import load_mat_dataset
from src.data.splits import make_holdout_split, make_kfold_splits, make_lomo_folds

ds_cfg = config.get('dataset', {})
dataset_path = ds_cfg.get('path')
print(f'Loading data from: {dataset_path}...')

keys_cfg   = ds_cfg.get('keys', {})
x_key      = ds_cfg.get('x_key', keys_cfg.get('x'))
y_key      = ds_cfg.get('y_key', keys_cfg.get('y'))
group_key  = ds_cfg.get('group_key', keys_cfg.get('groups'))
axis_key   = ds_cfg.get('axis_key', keys_cfg.get('axis'))
dataset_name = ds_cfg.get('name')

loaded = load_mat_dataset(
    dataset_path,
    x_key=x_key, y_key=y_key,
    group_key=group_key, axis_key=axis_key,
    dataset_name=dataset_name,
    keys=keys_cfg,
)
X_data, Y_data, x_axis, groups = loaded.X, loaded.y, loaded.x_axis, loaded.groups
N, L = X_data.shape
print(f'Dataset loaded: {N} spectra x {L} features. Classes: {np.unique(Y_data)}')

BATCH_SIZE = int(ds_cfg.get('batch_size', config.get('training', {}).get('batch_size', 32)))

split_cfg  = config.get('split', {})
SPLIT_SEED = int(split_cfg.get('seed', SEED))

# -- HOLDOUT SPLIT
holdout_cfg     = split_cfg.get('holdout', {})
test_size       = float(holdout_cfg.get('test_size', 0.20))
val_size_of_temp = float(holdout_cfg.get('val_size_of_temp', 0.50))
split_ho = make_holdout_split(Y_data, seed=SPLIT_SEED, test_size=test_size, val_size_of_temp=val_size_of_temp)

def make_loader(idx, shuffle):
    X_t = torch.from_numpy(X_data[idx]).unsqueeze(1).float()
    Y_t = torch.from_numpy(Y_data[idx])
    return DataLoader(TensorDataset(X_t, Y_t), batch_size=BATCH_SIZE, shuffle=shuffle)

train_loader_ho = make_loader(split_ho.idx_train, shuffle=True)
val_loader_ho   = make_loader(split_ho.idx_val,   shuffle=False)
test_loader_ho  = make_loader(split_ho.idx_test,  shuffle=False)

np.savez(
    os.path.join(output_dir, 'split_indices.npz'),
    idx_train=split_ho.idx_train, idx_val=split_ho.idx_val, idx_test=split_ho.idx_test,
    dataset_path=dataset_path, split_scheme='holdout'
)
print(f'Holdout | Train: {len(split_ho.idx_train)}, Val: {len(split_ho.idx_val)}, Test: {len(split_ho.idx_test)}')

# -- SECONDARY SPLITS
scheme = str(split_cfg.get('scheme', 'kfold')).lower()
if scheme == 'kfold':
    kfold_cfg = split_cfg.get('kfold', {})
    n_splits  = int(split_cfg.get('n_splits', kfold_cfg.get('n_splits', 5)))
    val_size  = float(split_cfg.get('val_size', kfold_cfg.get('val_size', 0.20)))
    CV_FOLDS  = make_kfold_splits(Y_data, seed=SPLIT_SEED, n_splits=n_splits, val_size=val_size)
    print(f'K-Fold: {len(CV_FOLDS)} folds')
elif scheme == 'lomo':
    lomo_cfg    = split_cfg.get('lomo', {})
    val_size    = float(lomo_cfg.get('val_size', split_cfg.get('val_size', 0.20)))
    skip_single = bool(lomo_cfg.get('skip_if_single_class_test', True))
    CV_FOLDS    = make_lomo_folds(Y_data, groups, seed=SPLIT_SEED, val_size=val_size, skip_if_single_class_test=skip_single)
    print(f'LOMO: {len(CV_FOLDS)} folds')
else:
    raise ValueError(f'Unsupported split scheme: {scheme}')

In [ ]:
# --- CELL 4: MODEL + FINETUNE SETTINGS ---
from src.models.factory import build_model_from_config
from src.engine.trainer import evaluate_model, train_one
from src.engine.finetune import train_linear_probe, train_finetune, compute_metrics
from src.utils.plotting import plot_fold_metrics, plot_aggregated_metrics

training_cfg = config.get('training', {})
pretrain_cfg = config.get('pretrain', {})

mode              = str(training_cfg.get('mode', 'finetune')).lower()
lp_epochs         = int(training_cfg.get('lp_epochs', training_cfg.get('linear_probe_epochs', 0)))
ft_epochs         = int(training_cfg.get('ft_epochs', training_cfg.get('epochs', 50)))
lr_lp             = float(training_cfg.get('lr_lp', training_cfg.get('learning_rate', 1e-3)))
lr_ft             = float(training_cfg.get('lr_ft', 1e-4))
weight_decay      = float(training_cfg.get('weight_decay', 0.0))
patience          = int(training_cfg.get('patience', 15))
scheduler_patience = int(training_cfg.get('scheduler_patience', 7))
grad_clip         = training_cfg.get('grad_clip', None)
if grad_clip is not None:
    grad_clip = float(grad_clip)
head_attr    = training_cfg.get('head_attr', 'classifier')
pretrained_path = pretrain_cfg.get('pretrained_path', training_cfg.get('pretrained_path'))
run_holdout  = bool(training_cfg.get('run_holdout', True))

finetune_dir = os.path.join(output_dir, 'finetune')
report_dir   = os.path.join(finetune_dir, 'report')
os.makedirs(report_dir, exist_ok=True)

print(f"Mode         : {mode}")
print(f"LP epochs    : {lp_epochs}  (lr={lr_lp})")
print(f"FT epochs    : {ft_epochs}  (lr={lr_ft})")
print(f"Pretrain path: {pretrained_path}")
print(f"Head attr    : {head_attr}")


# ---------------------------------------------------------------
# Weight loading: supports both direct match AND HybridMAE->HybridCNNTransformer
# (strips 'encoder.' prefix to map MAE checkpoint into the classifier model)
# ---------------------------------------------------------------
def load_pretrained_weights(model, path):
    if not path:
        print('⚠️  No pretrained path provided — training from scratch.')
        return False
    if not os.path.exists(path):
        print(f'⚠️  Pretrained path not found: {path}')
        return False

    checkpoint = torch.load(path, map_location=device)
    if isinstance(checkpoint, dict) and 'state_dict' in checkpoint:
        checkpoint = checkpoint['state_dict']

    # Attempt 1: direct load (same architecture)
    incompat = model.load_state_dict(checkpoint, strict=False)
    n_missing = len(incompat.missing_keys) if incompat.missing_keys else 0

    # Attempt 2: strip 'encoder.' prefix (HybridMAE -> HybridCNNTransformer)
    if n_missing > 3:
        encoder_state = {
            k.replace('encoder.', '', 1): v
            for k, v in checkpoint.items()
            if k.startswith('encoder.')
        }
        if encoder_state:
            incompat2 = model.load_state_dict(encoder_state, strict=False)
            n_missing2 = len(incompat2.missing_keys) if incompat2.missing_keys else 0
            print(f'✅ Loaded pretrained weights [encoder-strip mode]: {path}')
            print(f'   Missing keys after stripping: {n_missing2}')
            return True

    print(f'✅ Loaded pretrained weights [direct mode]: {path}')
    if incompat.missing_keys:
        print(f'   Missing keys  : {len(incompat.missing_keys)}')
    if incompat.unexpected_keys:
        print(f'   Unexpected keys: {len(incompat.unexpected_keys)}')
    return True


def build_model():
    return build_model_from_config(config, device)


def run_finetune_fold(train_loader, val_loader, test_loader, y_train, fold_name, save_dir):
    model = build_model()

    # Load pretrained weights when mode != scratch
    if mode in ('linear_probe', 'finetune'):
        load_pretrained_weights(model, pretrained_path)

    if mode == 'scratch':
        # Train entirely from random init
        best_state, history, _ = train_one(model, train_loader, val_loader, y_train, config, device)

    elif mode == 'linear_probe':
        # Stage 1 only: freeze encoder, train head
        if lp_epochs > 0:
            train_linear_probe(
                model, train_loader, device, y_train,
                head_attr=head_attr, epochs=lp_epochs,
                lr=lr_lp, weight_decay=weight_decay, grad_clip=grad_clip
            )
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        history = {'train_loss': [], 'val_loss': [], 'val_f1': [], 'val_acc': []}

    else:  # mode == 'finetune'
        # Stage 1: Linear Probe warm-up
        if lp_epochs > 0:
            print(f"  [Stage 1/2] Linear Probe ({lp_epochs} epochs, lr={lr_lp})")
            train_linear_probe(
                model, train_loader, device, y_train,
                head_attr=head_attr, epochs=lp_epochs,
                lr=lr_lp, weight_decay=weight_decay, grad_clip=grad_clip
            )

        # Stage 2: Full fine-tuning (all layers unlocked, lower LR)
        print(f"  [Stage 2/2] Full Fine-Tuning ({ft_epochs} epochs, lr={lr_ft})")
        best_state, history, _ = train_finetune(
            model, train_loader, val_loader, device, y_train,
            epochs=ft_epochs, lr=lr_ft, weight_decay=weight_decay,
            patience=patience, scheduler_patience=scheduler_patience,
            grad_clip=grad_clip
        )

    model.load_state_dict(best_state)
    preds, labels, probs, test_loss = evaluate_model(model, test_loader, device)
    metrics = compute_metrics(labels, preds, probs)
    metrics['fold']      = fold_name
    metrics['test_loss'] = float(test_loss)

    os.makedirs(save_dir, exist_ok=True)
    torch.save(model.state_dict(), os.path.join(save_dir, 'best_weights.pth'))
    return model, history, metrics, labels, preds, probs

In [ ]:
# --- CELL 5: HOLDOUT FINETUNE ---
holdout_metrics = None

if run_holdout:
    print('=' * 50)
    print(f'  HOLDOUT FINETUNE  ({ARCHITECTURE})  mode={mode}')
    print('=' * 50)

    holdout_dir = os.path.join(finetune_dir, 'holdout')
    model_ho, history_ho, holdout_metrics, labels_ho, preds_ho, probs_ho = run_finetune_fold(
        train_loader_ho, val_loader_ho, test_loader_ho,
        Y_data[split_ho.idx_train], 'holdout', holdout_dir
    )

    fig_path = os.path.join(report_dir, 'fig_holdout_metrics.png')
    plot_fold_metrics(history_ho, labels_ho, preds_ho, probs_ho, 'holdout', save_path=fig_path)

    ho_acc = accuracy_score(labels_ho, preds_ho)
    ho_f1  = f1_score(labels_ho, preds_ho, average='macro')
    print(f'Holdout Macro F1: {ho_f1:.4f} | Accuracy: {ho_acc:.4f}')
    print('Classification Report (Holdout)')
    print(classification_report(labels_ho, preds_ho, target_names=['HC (0)', 'IBD (1)']))

In [ ]:
# --- CELL 6: CV FINETUNE ---
import pandas as pd

folds_dir = os.path.join(finetune_dir, 'folds')
os.makedirs(folds_dir, exist_ok=True)

CV_RESULTS, all_labels, all_preds, all_probs = [], [], [], []

for fold in CV_FOLDS:
    fold_dir = os.path.join(folds_dir, fold.name)
    os.makedirs(fold_dir, exist_ok=True)

    y_train = Y_data[fold.idx_train]

    t_loader = make_loader(fold.idx_train, shuffle=True)
    v_loader = make_loader(fold.idx_val,   shuffle=False)
    te_loader = make_loader(fold.idx_test, shuffle=False)

    print(f'\n--- Running {fold.name} ---')
    model_kf, history_kf, metrics, labels, preds, probs = run_finetune_fold(
        t_loader, v_loader, te_loader, y_train, fold.name, fold_dir
    )

    CV_RESULTS.append(metrics)
    all_labels.append(labels)
    all_preds.append(preds)
    all_probs.append(probs)

    fig_path = os.path.join(report_dir, f'fig_{fold.name}_metrics.png')
    plot_fold_metrics(history_kf, labels, preds, probs, fold.name, save_path=fig_path)

# -- Aggregated stats
df = pd.DataFrame(CV_RESULTS)
print('\n=== Results per fold ===')
print(df.to_string(index=False))

f1s    = [m['macro_f1'] for m in CV_RESULTS]
accs   = [m['accuracy'] for m in CV_RESULTS]
aucs   = [m['roc_auc'] for m in CV_RESULTS if m.get('roc_auc') is not None]
praucs = [m['pr_auc']  for m in CV_RESULTS if m.get('pr_auc')  is not None]

summary = {
    'mean': {
        'accuracy': float(np.mean(accs))   if accs   else None,
        'macro_f1': float(np.mean(f1s))    if f1s    else None,
        'roc_auc' : float(np.mean(aucs))   if aucs   else None,
        'pr_auc'  : float(np.mean(praucs)) if praucs else None,
    },
    'std': {
        'accuracy': float(np.std(accs))   if accs   else None,
        'macro_f1': float(np.std(f1s))    if f1s    else None,
        'roc_auc' : float(np.std(aucs))   if aucs   else None,
        'pr_auc'  : float(np.std(praucs)) if praucs else None,
    },
}

scheme_name = config.get('split', {}).get('scheme', 'kfold').upper()
print(f'\nAggregated results {scheme_name} (mean ± std)')
print(f"Macro F1 : {summary['mean']['macro_f1']:.4f} ± {summary['std']['macro_f1']:.4f}")
print(f"Accuracy : {summary['mean']['accuracy']:.4f} ± {summary['std']['accuracy']:.4f}")
if summary['mean']['roc_auc'] is not None:
    print(f"ROC AUC  : {summary['mean']['roc_auc']:.4f} ± {summary['std']['roc_auc']:.4f}")
if summary['mean']['pr_auc'] is not None:
    print(f"PR AUC   : {summary['mean']['pr_auc']:.4f} ± {summary['std']['pr_auc']:.4f}")

labels_all = np.concatenate(all_labels) if all_labels else np.array([])
preds_all  = np.concatenate(all_preds)  if all_preds  else np.array([])
probs_all  = np.concatenate(all_probs)  if all_probs  else np.array([])

if labels_all.size > 0:
    agg_fig_path = os.path.join(report_dir, 'fig_global_metrics.png')
    plot_aggregated_metrics(labels_all, preds_all, probs_all, scheme_name, save_path=agg_fig_path)

metrics_path = os.path.join(finetune_dir, 'metrics_finetune.json')
with open(metrics_path, 'w') as f:
    json.dump({'holdout': holdout_metrics, 'cv': CV_RESULTS, 'summary': summary}, f, indent=2)
print(f'\n✅ Finetune done. Metrics saved to: {metrics_path}')